# Misja 14: zaobserwuj awarię i zmień plan

**Cel:** odrzucić nieaktualną akcję, zachować zaobserwowany stan i znaleźć plan naprawczy.
Wykonaj wcześniej 12 i 13: znamy już legalność kroków, koszty i ograniczenia baterii.
Awaria jest zdarzeniem zewnętrznym, a nie kolejną akcją wybraną przez planer.

To model symboliczny w Pythonie 3; komórki nie sterują robotem w Gazebo.
Uruchamiaj je kolejno. Zmieniamy tylko stan kuchenki: bateria i miejsce kelnera nie wracają do wartości początkowych.


In [ ]:
from pathlib import Path
import sys
# Działa z katalogu ćwiczeń, rozwiązań lub katalogu głównego serwera notebooków.
base = Path.cwd().resolve()
for candidate in (base, *base.parents, base / 'jupyter_notebooks'):
    if (candidate / 'robo_restaurant' / 'planning.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError('Otwórz ten notebook wewnątrz repozytorium')
from dataclasses import replace
from robo_restaurant.planning import State, Action, solve, execute, successors, goal


## Demonstracja: stary plan przestaje być wykonalny


In [ ]:
initial = State()
stale_plan, _ = solve(initial)
observed = initial
while stale_plan[0].name != 'prepare':
    observed = execute(observed, stale_plan.pop(0))
observed = replace(observed, cooker_ok=False)
print('Stan po zaobserwowaniu awarii kuchenki:', observed)
try:
    execute(observed, stale_plan[0])
except ValueError as error:
    print('Oczekiwane odrzucenie nieaktualnej akcji:', error)
else:
    raise AssertionError('Przygotowanie posiłku na uszkodzonej kuchence musi zostać odrzucone')

## Twoje zadanie

Napisz funkcję `recover(observed)`, zwracającą słownik z polami `status`, `plan` i `cost`.
Szukaj planu z **zaobserwowanego stanu**, a nie od początku poprzedniej misji.
Dla wykonalnego planu zwróć status `ready`; w przeciwnym razie `blocked`, z planem i kosztem `None`.
Przed zwróceniem wykonalnego planu odtwórz jego kroki i sprawdź osiągnięcie celu oraz koszt.

**Przewidź:** dlaczego ponowne uruchomienie starej akcji przygotowania nie naprawi uszkodzonej kuchenki?


In [ ]:
def recover(observed):
    # TODO: uzupełnij funkcję.
    raise NotImplementedError


## Sprawdź odpowiedź

Notebook w wersji początkowej uruchamia demonstrację, ale zadanie pozostaje oznaczone jako nieukończone.
Po uzupełnieniu funkcji komórka ma wypisać PASS. Przypadek bez planu ma pozostać `blocked`.


In [ ]:
try:
    result = recover(observed)
    assert result['status'] == 'ready' and result['cost'] == 28
    assert result['plan'][0] == Action('repair')
    state = observed
    for action in result['plan']:
        state = execute(state, action)
    assert goal(state)
    assert recover(State(location='table_1', battery=0, cooker_ok=False)) == {
        'status': 'blocked', 'plan': None, 'cost': None}
    print('Łączny koszt: przejazd przed awarią 2 + plan naprawczy 28 = 30')
except NotImplementedError:
    print("TODO: implement — uzupełnij funkcję; zadanie nie jest jeszcze zaliczone")
else:
    print("PASS")

## Od planu do wyniku wykonania

Który warunek przygotowania posiłku zmieniła awaria?
Dlaczego ponowne planowanie musi zachować baterię i położenie kelnera?
Co się zmienia, jeśli urządzenie może zepsuć się podczas trwającej akcji?

W drzewie z 11 **retry** powtarzało ten sam cel po ograniczonej naprawie.
Tutaj **replan** szuka nowej kolejności kroków z aktualnego stanu; może dodać naprawę lub ładowanie.
Oba mechanizmy mogą współpracować, ale nie są tym samym.

[Dalej: 15. Plan i executor](15.%20Planning%20and%20Execution%20-%20Bridge.ipynb) — czekamy na wynik,
zmieniamy stan dopiero po sukcesie, a po błędzie wybieramy retry, replan lub zatrzymanie.
